# Amazon Bedrock AgentCore Runtime e Agente com AgentCore Memory

## Visão Geral

Este tutorial demonstra como criar seu primeiro agente habilitado com memória usando AgentCore Runtime e AgentCore Memory. Você construirá um agente conversacional simples "Hello World" que lembra interações anteriores dentro de uma sessão.

### Detalhes do Tutorial


| Informação          | Detalhes                                                         |
|:--------------------|:-----------------------------------------------------------------|
| Tipo de tutorial    | Hello World / Introdução                                         |
| Tipo de agente      | Agente Conversacional Único                                      |
| Framework de agente | Strands Agents                                                   |
| Modelo LLM          | Anthropic Claude Haiku 4.5                                      |
| Recursos principais | AgentCore Runtime, Integração com Memory                         |
| Complexidade        | Iniciante                                                        |
| SDK utilizado       | boto3, bedrock-agentcore, bedrock-agentcore-starter-toolkit      |

### O Que Você Vai Aprender

Neste tutorial, você aprenderá:
1. Como criar um recurso de memória para seu agente
2. Como usar o AgentCoreMemorySessionManager para persistência automática de conversas
3. Como implantar seu agente no AgentCore Runtime
4. Como testar seu agente com gerenciamento de sessão


### Arquitetura

Este exemplo Hello World demonstra um agente conversacional simples implantado no AgentCore Runtime com integração de memória:

<div style="text-align:left">
    <img src="RuntimeMemoryIntegration.png" width="90%"/>
</div>


## 0. Pré-requisitos

Para executar este tutorial você precisará de:
* Python 3.10+
* Credenciais AWS configuradas
* Acesso ao modelo Amazon Bedrock (Claude Haiku 4.5)
* SDK do Amazon Bedrock AgentCore

Primeiro, vamos instalar as bibliotecas necessárias:

In [ ]:
!pip3 install -qr requirements.txt

### Configurando o Ambiente

Vamos importar as bibliotecas necessárias e configurar nosso ambiente:

In [ ]:
# Imports
import os
import boto3
import uuid
import logging
from bedrock_agentcore.memory import MemoryClient, MemorySessionManager
from bedrock_agentcore.memory.constants import ConversationalMessage, MessageRole

# Configuration
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)
logger = logging.getLogger("runtime-memory-agent")
REGION = os.getenv('AWS_REGION', 'us-west-2') # AWS region for the agent
memory_client = MemoryClient(region_name=REGION)

## 1. Criando o Recurso de Memória

Nesta seção, vamos criar um recurso de memória para nosso agente armazenar o histórico de conversas. A memória permite que o agente relembre interações passadas, mantenha o contexto e forneça respostas mais coerentes ao longo do tempo.

Para este exemplo, vamos criar um recurso simples de memória de curto prazo sem estratégias adicionais de longo prazo. A memória armazenará todas as mensagens de conversa, ajudando nosso agente a lembrar interações anteriores ao continuar uma sessão após ela ter sido encerrada no AgentCore Runtime.

In [ ]:
from botocore.exceptions import ClientError

# Create unique identifier for this resource
unique_id = str(uuid.uuid4())[:8]
memory_name = f"RuntimeMemoryAgent_{unique_id}"

try:
    # Create memory resource without strategies (short-term memory only)
    memory = memory_client.create_memory_and_wait(
        name=memory_name,
        strategies=[],  # No strategies for short-term memory
        description="Short-term memory for AgentCore Runtime agent",
        event_expiry_days=7, # Retention period for short-term memory
    )
    memory_id = memory['id']
    logger.info(f"✅ Created memory: {memory_id}")
except ClientError as e:
    logger.info(f"❌ ERROR: {e}")
    if e.response['Error']['Code'] == 'ValidationException' and "already exists" in str(e):
        # If memory already exists, retrieve its ID
        memories = memory_client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(memory_name)), None)
        logger.info(f"Memory already exists. Using existing memory ID: {memory_id}")
except Exception as e:
    # Show any errors during memory creation
    logger.error(f"❌ ERROR: {e}")
    import traceback
    traceback.print_exc()
    # Cleanup on error - delete the memory if it was partially created
    if 'memory_id' in locals() and memory_id:
        try:
            memory_client.delete_memory_and_wait(memory_id=memory_id)
            logger.info(f"Cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            logger.error(f"Failed to clean up memory: {cleanup_error}")

## 2. Criando Seu Agente Habilitado com Memória

Nesta seção, vamos construir nosso agente habilitado com memória usando o framework Strands Agents com o AgentCoreMemorySessionManager integrado para integração com memória.

> **Por Que a Memória é Importante**: Sessões no AgentCore Runtime expiram após um certo tempo, o que exclui o contexto da conversa. Ao armazenar conversas na memória, garantimos que informações anteriores persistam entre sessões, criando uma experiência contínua para os usuários mesmo após longos intervalos.

### Capacidades do Agente

Nosso agente irá:
1. Armazenar automaticamente cada mensagem do usuário e do assistente na memória
2. Recuperar o histórico de conversas anteriores ao continuar uma sessão existente
3. Manter o contexto em múltiplas interações com o mesmo usuário

### Componentes Principais da Nossa Implementação

#### 1. AgentCoreMemorySessionManager (Recomendado)
A integração nativa do Strands que lida automaticamente com:
- Armazenamento de cada mensagem do usuário e do assistente na memória
- Recuperação do histórico de conversas anteriores ao continuar uma sessão existente
- Gerenciamento do contexto de sessão e ator

#### 2. Inicialização do Agente
A função `initialize_agent`:
- Configura a memória usando `AgentCoreMemoryConfig` com memory_id, session_id e actor_id
- Cria uma instância de `AgentCoreMemorySessionManager`
- Configura o agente com o session_manager

#### 3. Handler do Ponto de Entrada
A função runtime_memory_agent:
- Analisa o payload de entrada e extrai a mensagem do usuário
- Gerencia a inicialização do agente e o rastreamento de sessão
- Lida com a invocação do agente com o contexto adequado
- Retorna respostas formatadas para o ambiente de runtime

Vamos criar o arquivo do nosso agente:

In [ ]:
%%writefile runtime_memory_agent.py
import os
import json
import logging
from typing import Dict, Any
from strands import Agent
from strands.models import BedrockModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig
from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager
# Configure detailed logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)
logger = logging.getLogger("runtime-memory-agent")

# Initialize the agent core app
app = BedrockAgentCoreApp()

MODEL_ID = os.getenv('MODEL_ID')
MEMORY_ID = os.getenv('MEMORY_ID')
REGION = os.getenv('AWS_REGION')

# Global agent instance - will be initialized with first request
agent = None
session_manager = None
 
   
    
def initialize_agent(actor_id, session_id):
    """Initialize the agent with memory session manager"""
    global agent, session_manager

    logger.info(f"Initializing agent for actor_id={actor_id}, session_id={session_id}")

    # Create model
    logger.info(f"Creating model with ID: {MODEL_ID}")
    model = BedrockModel(model_id=MODEL_ID)

    # Configure memory using AgentCoreMemoryConfig (recommended)
    logger.info(f"Creating memory config with region: {REGION}")
    config = AgentCoreMemoryConfig(
        memory_id=MEMORY_ID,
        session_id=session_id,
        actor_id=actor_id
    )

    # Create session manager - handles all memory operations automatically!
    session_manager = AgentCoreMemorySessionManager(config, region_name=REGION)

    # Create agent with session_manager (replaces custom hooks)
    logger.info("Creating agent with AgentCoreMemorySessionManager")
    agent = Agent(
        model=model,
        session_manager=session_manager,  # ✅ Built-in memory integration
        system_prompt="You're a helpful, memory-enabled agent deployed on AgentCore Runtime. You can remember previous interactions within the same session. Be friendly and concise in your responses."
    )
    logger.info("✅ Agent initialized with memory session manager")

@app.entrypoint
def runtime_memory_agent(payload, context):
    """
    Main entry point for the memory-enabled agent
    
    Args:
        payload: The input payload containing user data
        context: The runtime context object containing session information
    """
    global agent, session_manager
    
    # Log both payload and context info
    logger.info(f"Received payload: {payload}")
    logger.info(f"Context session_id: {context.session_id}")
    
    # Extract and validate required values
    user_input = payload.get("prompt")
    actor_id = payload.get("actor_id", "default_user")  # Provide default for demo
    session_id = context.session_id  # Get session_id from context
    
    # Validate required fields
    if user_input is None:
        error_msg = "❌ ERROR: Missing 'prompt' field in payload"
        logger.error(error_msg)
        return error_msg
    
    # Initialize agent on first request
    if agent is None:
        logger.info("First request - initializing agent")
        initialize_agent(actor_id, session_id)
    else:
        
        # Check if session or actor changed - need to reinitialize
        current_session = getattr(session_manager, '_session_id', None) if session_manager else None
        if current_session != session_id:
            logger.info(f"Session changed from {current_session} to {session_id} - reinitializing agent")
            initialize_agent(actor_id, session_id)
    
    # Invoke the agent with the user's input
    logger.info(f"Invoking agent with input: {user_input}")
    response = agent(user_input)
    response_text = response.message['content'][0]['text']
    logger.info(f"✅ Agent response: {response_text[:50]}...")
    
    return response_text

if __name__ == "__main__":
    logger.info("Starting AgentCore application")
    app.run()

## 3. Implantando no AgentCore Runtime

Nesta seção, vamos implantar nosso agente no Amazon Bedrock AgentCore Runtime, um ambiente de runtime gerenciado para agentes que oferece escalabilidade e operações simplificadas. O AgentCore Runtime cuida da complexidade da infraestrutura, permitindo que você se concentre na lógica do seu agente em vez de preocupações com implantação.

Diferente dos métodos tradicionais de implantação que exigem configuração e gerenciamento manual de servidores, o AgentCore Runtime empacota automaticamente seu código em containers, implanta-os na infraestrutura AWS e fornece endpoints HTTPS seguros para invocação. Essa abordagem garante que seu agente possa escalar conforme a demanda e operar de forma confiável em ambientes de produção.

### O Que Você Precisa Saber

- **AgentCore Runtime** empacota seu agente em um container Docker e o implanta na infraestrutura gerenciada da AWS
- **Variáveis de Ambiente** configurarão nosso agente:
  - `MEMORY_ID`: O recurso de memória que criamos anteriormente
  - `MODEL_ID`: ID do modelo Claude Haiku 4.5
  - `AWS_REGION`: Região AWS para implantação

> 💡 **Dica**: O AgentCore Starter Toolkit cuida de todas as etapas complexas de implantação para nós, incluindo IAM roles, repositórios ECR e builds de containers.

### Configurar a Implantação

Vamos configurar nossa implantação:

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
import time

agentcore_runtime = Runtime()
agent_name = f"runtime_memory_agent_{unique_id}"

response = agentcore_runtime.configure(
    entrypoint="runtime_memory_agent.py", 
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=REGION,
    agent_name=agent_name,
    memory_mode="STM_ONLY",
)
response


### Registrando Seu Recurso de Memória

No Passo 1, criamos manualmente um recurso de memória para entender os componentes subjacentes do AgentCore Memory. Como estamos gerenciando nosso próprio recurso de memória, precisamos atualizar a configuração local do starter toolkit para usá-lo durante a implantação. Isso evita que o toolkit provisione um novo recurso de memória no lançamento.

A célula a seguir escreve o ID do nosso recurso de memória no arquivo de configuração `.bedrock_agentcore.yaml` que foi gerado por `configure()`.

> **Nota**: Ao usar o AgentCore Starter Toolkit, você não precisa gerenciar recursos de memória manualmente. O AgentCore Starter Toolkit cuida do provisionamento, configuração e gerenciamento do ciclo de vida da memória automaticamente quando você define `memory_mode` durante `configure()`. Este tutorial cria o recurso explicitamente para que você possa ver como cada componente funciona de ponta a ponta.

In [ ]:
import yaml

config_path = ".bedrock_agentcore.yaml"
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Inject your existing memory_id into the agent's config
config['agents'][agent_name]['memory']['memory_id'] = memory_id

with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"✅ Injected memory_id: {memory_id} into {agent_name} config")

### Lançar o agente

Agora vamos lançar nosso agente no AgentCore Runtime. Esta etapa pega nosso agente configurado e o implanta na infraestrutura gerenciada do AgentCore. Durante esse processo, também estamos passando as variáveis de ambiente essenciais que nosso agente precisa: o ID da memória que criamos anteriormente e o ID do modelo a ser usado. Uma vez implantado, nosso agente estará acessível através de um endpoint seguro que podemos invocar com mensagens de usuário.

In [ ]:
launch_result = agentcore_runtime.launch(
    env_vars={
        "MEMORY_ID": memory_id,
        "MODEL_ID": "global.anthropic.claude-haiku-4-5-20251001-v1:0",
    }
)

### Verificar o status da implantação

Vamos verificar o status da implantação do nosso agente:

In [ ]:
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']

while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(f"Current status: {status}")

if status == 'READY':
    print("✅ Agent successfully deployed!")
else:
    print(f"❌ Deployment ended with status: {status}")

## Testando Seu Agente

Agora que nosso agente está implantado, vamos testá-lo enviando uma mensagem.

**Notas Importantes sobre Actor ID e Gerenciamento de Sessão**

- Actor ID: Em aplicações de produção, o actor_id normalmente viria do seu sistema de autenticação quando um usuário faz login. Esse identificador ajuda o agente a manter históricos de conversa separados para diferentes usuários. Em nosso exemplo, estamos usando um valor fixo (test_user_123), mas em cenários reais, você passaria o identificador único do usuário autenticado.

- Gerenciamento de Sessão: Embora o AgentCore Runtime gere automaticamente um ID de sessão se nenhum for fornecido, é recomendado gerenciar explicitamente os IDs de sessão na sua aplicação. Isso lhe dá melhor controle sobre:
  - Continuar conversas após expiração de sessão
  - Criar novas sessões quando apropriado (ex.: usuário inicia uma nova conversa)
  - Lidar com múltiplas conversas paralelas com o mesmo usuário
  - Implementar políticas de expiração de sessão baseadas nas necessidades da sua aplicação


In [ ]:
# Generate a test session ID
test_session_id = "agent-runtime-memory-session-123456789" # Min length is 33

# Send our first message
invoke_response = agentcore_runtime.invoke(
    {
        "prompt": "Hello! My name is John. What can you do?",
        "actor_id": "test_user_123"
    }, 
    session_id=test_session_id
)

invoke_response

### Exibir a resposta do agente

Vamos exibir a resposta em um formato mais legível:

In [ ]:
from IPython.display import Markdown, display
import json

response_text = invoke_response['response'][0]
display(Markdown(response_text))

### Testar persistência

Agora vamos testar se nosso agente lembra da interação anterior enviando uma mensagem de acompanhamento na mesma sessão:

In [ ]:
# Send a follow-up message using the same session ID
follow_up_response = agentcore_runtime.invoke(
    {
        "prompt": "What is my name?",
        "actor_id": "test_user_123"
    }, 
    session_id=test_session_id
)

# Display the response
follow_up_text = follow_up_response['response'][0]
display(Markdown(follow_up_text))

### Verificar conteúdo da memória

Vamos verificar o que está armazenado em nossa memória para confirmar que nossas mensagens foram salvas corretamente:

In [ ]:
# Use MemorySessionManager for session operations
manager = MemorySessionManager(memory_id=memory_id, region_name=REGION)
session = manager.create_memory_session(
    actor_id="test_user_123",
    session_id=test_session_id
)

# Get conversation history using session-scoped method
stored_turns = session.get_last_k_turns(k=10)

print(f"Found {len(stored_turns)} conversation turns in memory (shown in chronological order):")
for idx, turn in enumerate(stored_turns):
    print(f"\nTurn {idx+1}:")
    for message in turn:
        role = message['role']
        text = message['content']['text']
        print(f"- {role}: {text[:100]}...")

## Conceitos Principais

1. **Integração com Memória**: Como usar o Amazon Bedrock Memory para armazenar histórico de conversas
2. **Gerenciamento de Sessão**: Como usar IDs de sessão para manter o contexto da conversa
3. **Implantação no AgentCore**: Como implantar seu agente em um ambiente de runtime de produção
4. **AgentCoreMemorySessionManager**: Como usar a integração nativa do Strands para gerenciamento automático de memória

Esses conceitos fornecem uma base para construir agentes mais complexos com memória persistente.

## Limpeza (Opcional)

Se você não precisa mais dos recursos criados neste tutorial, pode limpá-los:

In [ ]:
# Get resource identifiers
if 'launch_result' in locals():
    print(f"Agent ID: {launch_result.agent_id}")
    print(f"ECR Repository: {launch_result.ecr_uri.split('/')[1]}")
else:
    print("Launch results not available")

In [ ]:
# Only run this cell if you want to delete the resources

# Delete the AgentCore Runtime
agentcore_control_client = boto3.client(
    'bedrock-agentcore-control',
    region_name=REGION
)

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id,
)
print(f"Deleted AgentCore Runtime: {launch_result.agent_id}")

# Delete the ECR repository
ecr_client = boto3.client(
    'ecr',
    region_name=REGION
)

response = ecr_client.delete_repository(
    repositoryName=launch_result.ecr_uri.split('/')[1].split(':')[0],
    force=True
)
print(f"Deleted ECR repository: {launch_result.ecr_uri.split('/')[1].split(':')[0]}")

# Delete the memory resource
memory_client = MemoryClient(region_name=REGION)
memory_client.delete_memory_and_wait(memory_id=memory_id)
print(f"Deleted memory resource: {memory_id}")

## Parabéns!

Você construiu e implantou com sucesso seu primeiro agente habilitado com memória usando Amazon Bedrock AgentCore Runtime e AgentCore Memory!

### Próximos Passos

Agora que você entende o básico, você pode:

1. **Adicionar Ferramentas**: Aprimore seu agente com ferramentas como calculadoras, conectores de banco de dados ou chamadas de API
2. **Melhorar a Memória**: Implemente estratégias de memória mais sofisticadas com memória de longo prazo
3. **Construir uma UI**: Crie uma interface web ou mobile para seu agente